In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import concurrent.futures
import os

In [ ]:
# ==========================================
# 0. CONFIGURATION: DP HYPERPARAMETER SWEEP
# ==========================================
# Format: (Width, Use_DP, Epochs, Batch_Size, Clip_Norm, Noise_Mult)
RUN_CONFIG = [
    (50, False, 100000, 128, 0.0,  0.0),  # 1. Pure SGD Baseline
    (50, True, 100000, 128, 1.0,  1.0),  # 1. Pure SGD Baseline
    (50, True, 100000, 128, 0.5,  1.0),  # 1. Pure SGD Baseline
    (50, True, 100000, 128, 1.0,  3.0),  # 1. Pure SGD Baseline
    (500, False, 100000, 128, 0.0,  0.0),  # 1. Pure SGD Baseline
    (500, True, 100000, 128, 1.0,  1.0),  # 1. Pure SGD Baseline
    (500, True, 100000, 128, 0.5,  1.0),  # 1. Pure SGD Baseline
    (500, True, 100000, 128, 1.0,  3.0),  # 1. Pure SGD Baseline
    (5000, False, 100000, 128, 0.0,  0.0),  # 1. Pure SGD Baseline
    (5000, True, 100000, 128, 1.0,  1.0),  # 1. Pure SGD Baseline
    (5000, True, 100000, 128, 0.5,  1.0),  # 1. Pure SGD Baseline
    (5000, True, 100000, 128, 1.0,  3.0),  # 1. Pure SGD Baseline
]

LR = 0.001


# ==========================================
# 1. NTK Architecture
# ==========================================
class NTKLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))
        self.c = in_features ** 0.5 

    def forward(self, x):
        return F.linear(x, self.weight, self.bias) / self.c

class WideMLP(nn.Module):
    def __init__(self, input_dim, width, output_dim):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            NTKLinear(input_dim, width),
            nn.ReLU()
        )
        self.final_layer = NTKLinear(width, output_dim)

    def forward(self, x):
        feat = self.feature_extractor(x)
        return self.final_layer(feat), feat

# ==========================================
# 2. Data Setup (100-dim MNIST)
# ==========================================
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

N_train, N_test = 2000, 800
def process_mnist(dataset, N):
    imgs = dataset.data[:N].unsqueeze(1).float() / 255.0
    imgs_pooled = F.adaptive_avg_pool2d(imgs, (10, 10))
    X = imgs_pooled.view(N, -1)
    Y = dataset.targets[:N].long()
    return X, Y

X_train_cpu, Y_train_cpu = process_mnist(mnist_train, N_train)
X_test_cpu, Y_test_cpu = process_mnist(mnist_test, N_test)
train_dataset = TensorDataset(X_train_cpu, Y_train_cpu)

# ==========================================
# 3. FAST Training Worker (vmap DP-SGD)
# ==========================================
def train_worker(args):
    width, use_dp, custom_epochs, batch_size, clip_norm, noise_mult, gpu_id = args
    device = f'cuda:{gpu_id}'
    
    # FIXED: The label now explicitly includes the Width so dictionary keys are perfectly unique
    if use_dp:
        label = f"W:{width} DP (C:{clip_norm} N:{noise_mult})"
    else:
        label = f"W:{width} SGD"
        
    print(f"-> Starting {label} on {device}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=32, pin_memory=True,persistent_workers=True)
    X_te, Y_te = X_test_cpu.to(device), Y_test_cpu.to(device)

    model = WideMLP(100, width, 10).to(device)
    init_params = [p.clone().detach() for p in model.parameters()]
    total_init_norm = sum(torch.sum(p**2).item() for p in init_params)**0.5
    
    with torch.no_grad():
        _, init_features_test = model(X_te)
        init_features_test = init_features_test.clone().detach()
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    h = {'ep': [], 'loss': [], 'acc': [], 'dist': [], 'f_shift': [], 'max_w': []}

    for ep in range(custom_epochs + 1):
        model.train()
        epoch_loss, total_samples = 0.0, 0
        
        for X_b, Y_b in train_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            optimizer.zero_grad()
            
            if not use_dp:
                out, _ = model(X_b)
                loss = criterion(out, Y_b)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * len(X_b)
            else:
                params = dict(model.named_parameters())
                def compute_loss(params, x, y):
                    out, _ = torch.func.functional_call(model, params, (x.unsqueeze(0),))
                    return criterion(out, y.unsqueeze(0))
                
                per_sample_grads, batch_losses = torch.func.vmap(
                    torch.func.grad_and_value(compute_loss), in_dims=(None, 0, 0)
                )(params, X_b, Y_b)
                
                epoch_loss += batch_losses.sum().item()
                
                for p_name, p in model.named_parameters():
                    g = per_sample_grads[p_name] 
                    g_flat = g.view(len(X_b), -1)
                    norms = g_flat.norm(dim=1)
                    
                    clip_coef = torch.clamp(clip_norm / (norms + 1e-6), max=1.0)
                    
                    for _ in range(g.dim() - 1):
                        clip_coef = clip_coef.unsqueeze(-1)
                    
                    p.grad = (g * clip_coef).mean(dim=0) + \
                             (noise_mult * clip_norm / len(X_b)) * torch.randn_like(p)
                optimizer.step()
            total_samples += len(X_b)

        if ep % max(1, custom_epochs // 1000) == 0 or ep == custom_epochs:
            model.eval()
            with torch.no_grad():
                out_te, current_features_test = model(X_te)
                acc = (out_te.argmax(1) == Y_te).float().mean().item()
                dist = sum(torch.sum((p-pi)**2).item() for p, pi in zip(model.parameters(), init_params))**0.5
                max_w = max(torch.max(torch.abs(p-pi)).item() for p, pi in zip(model.parameters(), init_params))
                f_shift = torch.norm(current_features_test - init_features_test, dim=1).mean().item()

                if use_dp:
                    label = f"W:{width} DP (C:{clip_norm} N:{noise_mult}) | Epoch {ep:<1} | Train Loss {epoch_loss / total_samples:.2f} | Acc: {acc*100:.1f}%"
                else:
                    label = f"W:{width} SGD | Epoch {ep:<1} | Train Loss {epoch_loss / total_samples:.2f} | Acc: {acc*100:.1f}%"
                    
                print(f"-- Training {label}")
                    
                h['ep'].append(ep); h['loss'].append(epoch_loss / total_samples); h['acc'].append(acc)
                h['dist'].append(dist/total_init_norm); h['f_shift'].append(f_shift); h['max_w'].append(max_w)

    print(f"<- Finished {label}")
    return label, h

# ==========================================
# 4. Dispatch, Save, and Plot
# ==========================================
jobs = [(cfg[0], cfg[1], cfg[2], cfg[3], cfg[4], cfg[5], i % 2) for i, cfg in enumerate(RUN_CONFIG)]
results = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as ex:
    for job, (label, h) in zip(jobs, ex.map(train_worker, jobs)):
        results[label] = {
            'width': job[0],
            'use_dp': job[1],
            'history': h
        }

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 11)) # Slightly taller to fit legend

# 1. Base color map
unique_widths = sorted(list(set([res['width'] for res in results.values()])))
color_palette = ['blue', 'red', 'green', 'purple', 'orange', 'brown']
width_to_color = {w: color_palette[i % len(color_palette)] for i, w in enumerate(unique_widths)}

# 2. Line style map
dp_linestyles = ['--', ':', '-.', (0, (3, 5, 1, 5)), (0, (5, 10))] 
width_dp_counter = {w: 0 for w in unique_widths}

lines = []
labels = []

for label, res in results.items():
    w = res['width']
    use_dp = res['use_dp']
    h = res['history']
    c = width_to_color[w]
    
    if not use_dp:
        ls, lw = '-', 2.5
    else:
        ls = dp_linestyles[width_dp_counter[w] % len(dp_linestyles)]
        width_dp_counter[w] += 1
        lw = 2.0
    
    # Store the line object from the first plot to build the global legend
    line, = axs[0, 0].plot(h['ep'], h['loss'], color=c, linestyle=ls, linewidth=lw)
    if label not in labels:
        lines.append(line)
        labels.append(label)

    # Plot the rest without adding to labels again
    axs[0, 1].plot(h['ep'], h['acc'], color=c, linestyle=ls, linewidth=lw)
    axs[0, 2].plot(h['ep'], h['dist'], color=c, linestyle=ls, linewidth=lw)
    axs[1, 0].plot(h['ep'], h['f_shift'], color=c, linestyle=ls, linewidth=lw)
    axs[1, 1].plot(h['ep'], h['max_w'], color=c, linestyle=ls, linewidth=lw)

# Titles
axs[0, 0].set_title('1. Train Loss (CrossEntropy)')
axs[0, 1].set_title('2. Test Accuracy')
axs[0, 2].set_title('3. Lazy Training: Total Param Movement')
axs[1, 0].set_title('4. Feature Shift (No Feature Learning)')
axs[1, 1].set_title('5. Max Single Weight Shift (L-infinity)')
axs[1, 2].axis('off')

# Formatting clean up
for ax in axs.flat:
    ax.grid(True, alpha=0.3)
    if ax != axs[1, 2]: # Don't label the empty plot
        ax.set_xlabel("Epochs")

# --- GLOBAL LEGEND AT BOTTOM ---
# adjust bottom to make room (0.15 is roughly 15% of the figure height)
plt.subplots_adjust(bottom=0.15, hspace=0.3) 

fig.legend(
    lines, 
    labels, 
    loc='lower center', 
    bbox_to_anchor=(0.5, 0.02), 
    ncol=3,             # Arrange legend in 3 columns to save vertical space
    fontsize='medium',
    frameon=True
)

# ==========================================
# 5. Save Results
# ==========================================
os.makedirs("results", exist_ok=True)
plt.savefig("results/ntk_dp_comparison1000slow.png", dpi=300, bbox_inches='tight')
plt.savefig("results/ntk_dp_comparison1000slow.pdf", bbox_inches='tight')
torch.save(results, "results/ntk_dp_data1000slow.pt")

print(f"\n✅ Run complete! Generated {len(labels)} unique curves.")
plt.show()